# Agentes con LlamaIndex

In [1]:
%pip install llama-index requests

### Configurar el LLM y el modelo de embeddings

In [2]:
import nest_asyncio

nest_asyncio.apply()

import os

os.environ["OPENAI_API_KEY"] = ""

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

llm = OpenAI(model="gpt-4", temperature=0.1)
embed_model = OpenAIEmbedding()

Settings.llm = llm
Settings.embed_model = embed_model

### Crear herramientas

In [5]:
# Importaciones de librerías a utilizar
import requests
from datetime import datetime
try:
    from zoneinfo import ZoneInfo
except ImportError:
    ZoneInfo = None

from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import (
    FunctionAgent,              # Importantes herramientas
    ReActAgent,                 # Importantes herramientas
)

from IPython.display import display, HTML
import xml.etree.ElementTree as ET


In [9]:
# Funciones matemáticas:

def multiply(a: int, b: int) -> int:
    """Multiplica dos números enteros y devuelve el resultado."""
    return a * b

def add(a: int, b: int) -> int:
    """Suma dos enteros y devuelve el resultado."""
    return a + b

def subtract(a: int, b: int) -> int:
    """Resta dos enteros y devuelve el resultado."""
    return a - b

def divide(a: float, b: float) -> float:
    """Divide dos números y devuelve el resultado."""
    if b == 0:
        return float("inf")
    return a / b

def power(a: float, b: float) -> float:
    """Eleva un número a la potencia deseada."""
    return a ** b

In [10]:
# Declarar las funciones en el lenguaje de LlamaIndex
multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)
subtract_tool = FunctionTool.from_defaults(fn=subtract)
divide_tool = FunctionTool.from_defaults(fn=divide)
power_tool = FunctionTool.from_defaults(fn=power)

### Crear un ReAct Agent

In [11]:
agent = ReActAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
    ],
    llm=llm,)

In [12]:
resultado = await agent.run("Dime el resultado de 507 + 3^4 + (45/3)")

In [16]:
resultado.response

ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='El resultado de 507 + 3^4 + (45/3) es 603.')])

### Funciones para revelar el trace

In [17]:
from llama_index.core.agent.workflow import ToolCallResult, ToolCall

async def run_with_trace(agent, query: str):
    """Ejecuta una consulta y devuelve (respuesta, trace).

    El trace es una lista de diccionarios con la tool llamada, los
    argumentos y la salida (truncada a 300 caracteres).
    """
    handler = agent.run(query)
    trace = []
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            salida = str(event.tool_output)
            if len(salida) > 300:
                salida = salida[:300] + "..."
            trace.append({
                "tool": event.tool_name,
                "args": dict(event.tool_kwargs),
                "output": salida,
            })
    response = await handler
    return response, trace


def mostrar_trace(query, response, trace):
    html = [f'<div style="font-family:sans-serif;border:1px solid #ccc;padding:10px;margin:8px 0">']
    html.append(f'<p style="font-size:16px"><strong>Consulta:</strong> {query}</p>')
    html.append(f'<p style="font-size:16px"><strong>Respuesta:</strong> {response}</p>')
    if trace:
        html.append('<p style="font-size:15px"><strong>Trace de tools:</strong></p><ol>')
        for paso in trace:
            html.append(
                f'<li><code>{paso["tool"]}</code>('
                f'{paso["args"]}) &rarr; <em>{paso["output"]}</em></li>'
            )
        html.append('</ol>')
    else:
        html.append('<p><em>El agente respondió sin invocar tools.</em></p>')
    html.append('</div>')
    display(HTML(''.join(html)))

In [18]:
query = "Calcula (12 * 7) + 2^10 y dime el resultado final."
response, trace = await run_with_trace(agent, query)
mostrar_trace(query, response, trace)

### Funciones avanzadas



In [21]:
# Conversión de monedas con la API de FrankFurter

def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convierte moneda usando la API Frankfurter (BCE, sin API key)."""
    url = "https://api.frankfurter.app/latest"
    params = {
        "amount": amount,
        "from": from_currency.upper(),
        "to": to_currency.upper(),
    }
    try:
        data = requests.get(url, params=params, timeout=10).json()
        rate = data.get("rates", {}).get(to_currency.upper())
        if rate is None:
            return f"No fue posible convertir {from_currency} a {to_currency}."
        return f"{amount} {from_currency.upper()} = {rate:.2f} {to_currency.upper()}"
    except Exception as exc:
        return f"Error en conversión de moneda: {exc}"

In [26]:
# Declarar las funciones en el lenguaje de LlamaIndex
convert_currency_tool = FunctionTool.from_defaults(fn=convert_currency)

agent_currency = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        convert_currency_tool,
    ],

    llm=llm,)

In [32]:
query = "¿Cuánto son 40 dólares americanos en libras esterlinas?"
response, trace = await run_with_trace(agent_currency, query)
mostrar_trace(query, response, trace)

In [33]:
# Búsqueda de información en Wikipedia
def search_web(query: str) -> str:
    """Busca un resumen en Wikipedia (es, con fallback a en)."""
    headers = {"User-Agent": "research-agent/1.0 (educational use)"}
    for lang in ("es", "en"):
        try:
            search_url = f"https://{lang}.wikipedia.org/w/api.php"
            params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": 1,
            }
            r = requests.get(search_url, params=params, headers=headers, timeout=10)
            hits = r.json().get("query", {}).get("search", [])
            if not hits:
                continue
            title = hits[0]["title"]
            summary_url = (
                f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/"
                f"{requests.utils.quote(title)}"
            )
            s = requests.get(summary_url, headers=headers, timeout=10)
            if s.status_code == 200:
                extract = s.json().get("extract")
                if extract:
                    return f"[Wikipedia/{lang}] {extract}"
        except Exception as exc:
            return f"Error en búsqueda web: {exc}"
    return f"No se encontró información para '{query}'."


In [49]:
# Declarar las funciones en el lenguaje de LlamaIndex
search_web_tool = FunctionTool.from_defaults(fn=search_web)

agent_search = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        convert_currency_tool,
        search_web_tool,
    ],

    llm=llm,)

In [50]:
query = "Busca cuál es el tamaño de la población de Colombia y dime cuánto sería si aumenta en un 15%"
response, trace = await run_with_trace(agent_search, query)
mostrar_trace(query, response, trace)

In [53]:
# Obtener datos del clima

def get_weather(location: str = "Madrid") -> str:
    """Obtiene el clima actual usando wttr.in."""
    try:
        url = f"https://wttr.in/{location}"
        params = {"format": "3"}
        result = requests.get(url, params=params, timeout=10)
        if result.status_code == 200:
            return result.text.strip()
        return f"No se pudo obtener el clima para {location}."
    except Exception as exc:
        return f"Error al obtener clima: {exc}"


In [54]:
# Declarar las funciones en el lenguaje de LlamaIndex
weather_tool = FunctionTool.from_defaults(fn=get_weather)

agent_weather = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        weather_tool
    ],

    llm=llm,)

In [55]:
query = "Dime cómo va a estar el clima mañana en Bogotá, Colombia"
response, trace = await run_with_trace(agent_weather, query)
mostrar_trace(query, response, trace)

In [56]:
# Obtener las últimas noticias

def get_news(category: str = "world") -> str:
    """Obtiene las últimas noticias vía RSS de Google News (sin API key)."""
    topic_map = {
        "world": "WORLD",
        "business": "BUSINESS",
        "technology": "TECHNOLOGY",
        "sports": "SPORTS",
        "science": "SCIENCE",
        "health": "HEALTH",
        "entertainment": "ENTERTAINMENT",
    }
    topic = topic_map.get(category.lower(), "WORLD")
    url = f"https://news.google.com/rss/headlines/section/topic/{topic}?hl=es&gl=ES&ceid=ES:es"
    try:
        result = requests.get(url, timeout=10)
        root = ET.fromstring(result.content)
        items = root.findall(".//item")[:5]
        if not items:
            return "No se encontraron noticias."
        lines = [f"- {item.findtext('title', 'Sin título')}" for item in items]
        return "\n".join(lines)
    except Exception as exc:
        return f"Error al obtener noticias: {exc}"


In [57]:
# Declarar las funciones en el lenguaje de LlamaIndex
news_tool = FunctionTool.from_defaults(fn=get_news)

agent_news = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        news_tool
    ],

    llm=llm,)

In [59]:
query = "¿Cuáles son las últimas noticias en el mundo de la Inteligencia Artificial?"
response, trace = await run_with_trace(agent_news, query)
mostrar_trace(query, response, trace)

In [61]:
# Conversión de horas
def current_time(timezone: str = "UTC") -> str:
    """Devuelve la hora actual en la zona horaria solicitada."""
    if ZoneInfo is None:
        now = datetime.utcnow()
        return f"Hora actual UTC: {now.strftime('%Y-%m-%d %H:%M:%S')} (ZoneInfo no disponible)"
    try:
        tz = ZoneInfo(timezone)
        now = datetime.now(tz)
        return now.strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception:
        now = datetime.utcnow()
        return f"Zona horaria inválida. Hora UTC: {now.strftime('%Y-%m-%d %H:%M:%S')}"

time_tool = FunctionTool.from_defaults(fn=current_time)


In [62]:
agent_time = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        time_tool
    ],

    llm=llm,)

In [63]:
query = "¿Qué hora es en Beijing?"
response, trace = await run_with_trace(agent_time, query)
mostrar_trace(query, response, trace)

In [74]:
# Importar
import math
import scipy

def calcular_integral (expression: str, limite_inferior: float, limite_superior) -> float:
  """ Calcular la integral definida de una función matemática de una variable (x)

  Args:
    expression (str): La función matemática a integrar (ejemplo: 'x**2 + 2*x + 1')
    limite_inferior (float): El límite inferior de la integral
    limite_superior (float): El límite superior de la integral

  Returns:
    float: El valor de la integral definida de la función en el intervalo dado

  """
  # Definir la funión interna que Scipy va a integrar
  def f(x):
    # Crear un entorno seguro que incluya 'math' y la variable 'x'
    entorno = {"math": math, "x": x}

    # Evluar el string envido por el LLM
    return eval(expression, entorno)

  # Calcular la integral definida
  integral = scipy.integrate.quad(f, limite_inferior, limite_superior)

  return integral[0]

# Llevarla al lenguaje de LlamaIndex

integral_tool = FunctionTool.from_defaults(fn=calcular_integral)

In [75]:
agent_integrales = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        integral_tool
    ],

    llm=llm,)

In [76]:
query = "Cuál es el valor de la integral de 0 a 3 de la función x^2 + 2x + 1"
response, trace = await run_with_trace(agent_integrales, query)
mostrar_trace(query, response, trace)

## Agente resumen con todas las funciones implementadas
- Operaciones de aritmética básica
- Búsquedas en la base de datos de Wikipedia
- Conversión de monedas (Algunas)
- Conversión de horas
- Consulta del clima

In [64]:
agent_time = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
        search_web_tool,
        convert_currency_tool,
        weather_tool,
        news_tool,
        time_tool
    ],

    llm=llm,)

In [65]:
query = "¿Qué hora es en Beijing y cuál va a ser el clima durante las próximas horas en esa ciudad? Necesito saber a cómo está el tipo de cambio con respecto al dolar"
response, trace = await run_with_trace(agent_time, query)
mostrar_trace(query, response, trace)

In [66]:
agent_basic = FunctionAgent(
    tools=[
        multiply_tool,
        add_tool,
        subtract_tool,
        divide_tool,
        power_tool,
    ],

    llm=llm,)

In [67]:
query = "¿Qué hora es en Beijing y cuál va a ser el clima durante las próximas horas en esa ciudad? Necesito saber a cómo está el tipo de cambio con respecto al dolar"
response, trace = await run_with_trace(agent_basic, query)
mostrar_trace(query, response, trace)

In [70]:
query = "Cuánto es 5 ^ 5 - 70 + 90*7"
response, trace = await run_with_trace(agent_basic, query)
mostrar_trace(query, response, trace)